In [1]:
#testing

StatementMeta(, ff9c28aa-7efe-4e03-be59-9f61e396e84b, 3, Finished, Available, Finished, False)

In [ ]:
import requests
import json
import time

auth = json.loads(mssparkutils.notebook.run("procore_auth"))
token = auth["token"]
COMPANY_ID = auth["company_id"]
headers = {
    "Authorization": f"Bearer {token}",
    "Procore-Company-Id": str(COMPANY_ID)
}

print("Auth successful" if token else "Auth failed")

StatementMeta(, ff9c28aa-7efe-4e03-be59-9f61e396e84b, 4, Finished, Available, Finished, False)

Auth successful


In [3]:
projects_response = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID}
)
projects = projects_response.json()
print(f"{len(projects)} projects found" if isinstance(projects, list) else "Failed to fetch projects")

StatementMeta(, ff9c28aa-7efe-4e03-be59-9f61e396e84b, 5, Finished, Available, Finished, False)

18 projects found


In [4]:
all_requisitions = []

for project in projects:
    project_id = project["id"]
    project_name = project["name"]

    page = 1
    while True:
        response = requests.get(
            "https://api.procore.com/rest/v1.1/requisitions",
            headers=headers,
            params={
                "project_id": project_id,
                "page": page,
                "per_page": 100
            }
        )

        if response.status_code != 200:
            break

        rows = response.json()

        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_requisitions.extend(rows)

        if len(rows) < 100:
            break

        page += 1
        time.sleep(0.3)

print(f"Total requisitions found: {len(all_requisitions)}")

StatementMeta(, ff9c28aa-7efe-4e03-be59-9f61e396e84b, 6, Finished, Available, Finished, False)

Total requisitions found: 651


In [5]:
import collections

# Filter to unpaid requisitions only
unpaid_reqs = [
    r for r in all_requisitions
    if not r.get("payment_summary", {}).get("invoice_paid_in_full", False)
]

# Show breakdown
status_breakdown = collections.Counter([r.get("status") for r in unpaid_reqs])

print(f"Total requisitions: {len(all_requisitions)}")
print(f"Unpaid requisitions: {len(unpaid_reqs)}")
print(f"Paid requisitions: {len(all_requisitions) - len(unpaid_reqs)}")
print(f"\nUnpaid status breakdown:")
for status, count in status_breakdown.items():
    print(f"  {status}: {count}")

StatementMeta(, ff9c28aa-7efe-4e03-be59-9f61e396e84b, 7, Finished, Available, Finished, False)

Total requisitions: 651
Unpaid requisitions: 273
Paid requisitions: 378

Unpaid status breakdown:
  approved: 79
  under_review: 79
  draft: 9
  pending_owner_approval: 104
  revise_and_resubmit: 2


In [6]:
all_line_items = []

print(f"Pulling line items for {len(unpaid_reqs)} unpaid requisitions...")

for req in unpaid_reqs:
    req_id = req["id"]
    project_id = req["project_id"]
    project_name = req["project_name"]
    req_number = req.get("number", "")
    req_status = req.get("status", "")
    vendor_id = req.get("vendor_id", "")
    vendor_name = req.get("vendor_name", "")
    commitment_id = req.get("commitment_id", "")
    billing_date = req.get("billing_date", "")
    invoice_number = req.get("invoice_number", "")
    total_claimed = req.get("total_claimed_amount", "")
    amount_due = req.get("payment_summary", {}).get("invoiced_amount_due", "")

    max_retries = 3
    retry_count = 0

    while retry_count < max_retries:
        response = requests.get(
            f"https://api.procore.com/rest/v1.0/requisitions/{req_id}/contract_detail_items",
            headers=headers,
            params={"project_id": project_id}
        )

        if response.status_code == 429:
            print(f"  Rate limited — waiting 60 seconds...")
            time.sleep(60)
            retry_count += 1
            continue

        if response.status_code != 200:
            print(f"  Error {response.status_code} for requisition {req_id}, skipping")
            break

        rows = response.json()

        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["requisition_id"] = req_id
            row["requisition_number"] = req_number
            row["requisition_status"] = req_status
            row["vendor_id"] = vendor_id
            row["vendor_name"] = vendor_name
            row["commitment_id"] = commitment_id
            row["billing_date"] = billing_date
            row["invoice_number"] = invoice_number
            row["total_claimed_amount"] = total_claimed
            row["amount_due"] = amount_due
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_line_items.extend(rows)
        break

    time.sleep(2)

print(f"Done! Total line items: {len(all_line_items)}")

StatementMeta(, ff9c28aa-7efe-4e03-be59-9f61e396e84b, 8, Finished, Cancelled, Cancelled, False)

Pulling line items for 273 unpaid requisitions...
  Error 404 for requisition 562949958136384, skipping
  Error 404 for requisition 562949958186183, skipping
  Error 404 for requisition 562949958514300, skipping
  Error 404 for requisition 562949958514310, skipping
  Error 404 for requisition 562949958606832, skipping
  Error 404 for requisition 562949958620144, skipping
  Error 404 for requisition 562949958772199, skipping
  Error 404 for requisition 562949958772304, skipping
  Error 404 for requisition 562949958772315, skipping
  Error 404 for requisition 562949958801491, skipping
  Error 404 for requisition 562949959022574, skipping
  Error 404 for requisition 562949959285475, skipping
  Error 404 for requisition 562949959287447, skipping
  Error 404 for requisition 562949959380717, skipping
  Error 404 for requisition 562949959536112, skipping
  Error 404 for requisition 562949959617258, skipping
  Error 404 for requisition 562949959826882, skipping
  Error 404 for requisition 5629

In [ ]:
print(f"Total line items returned: {len(all_line_items)}")
print(f"Unique requisitions with data: {len(set([item['requisition_id'] for item in all_line_items]))}")
print(f"Requisitions skipped (404): {sum(1 for r in unpaid_reqs if not any(item['requisition_id'] == r['id'] for item in all_line_items))}")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# Find which projects have blocked requisitions
blocked_reqs = [
    r for r in unpaid_reqs
    if not any(item["requisition_id"] == r["id"] for item in all_line_items)
]

import collections
blocked_by_project = collections.Counter([r["project_name"] for r in blocked_reqs])

print(f"Blocked requisitions by project:")
for project, count in sorted(blocked_by_project.items(), key=lambda x: x[1], reverse=True):
    print(f"  {project}: {count}")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
response = requests.get(
    "https://api.procore.com/rest/v1.1/requisitions",
    headers=headers,
    params={
        "project_id": projects[0]["id"],
        "page": 1,
        "per_page": 3
    }
)

data = response.json()
if data:
    print(list(data[0].keys()))
    print(data[0].get("payment_summary"))

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from datetime import datetime, timedelta

# Calculate 12 months ago
twelve_months_ago = datetime.now() - timedelta(days=365)

# Filter requisitions from last 12 months
recent_reqs = [
    r for r in all_requisitions 
    if r.get("updated_at") and 
    datetime.strptime(r["updated_at"][:10], "%Y-%m-%d") >= twelve_months_ago
]

# Filter by active statuses only
active_statuses = ["under_review", "pending_owner_approval", "revise_and_resubmit", "draft"]
active_recent_reqs = [r for r in recent_reqs if r.get("status") in active_statuses]

print(f"Total requisitions: {len(all_requisitions)}")
print(f"Requisitions from last 12 months: {len(recent_reqs)}")
print(f"Active requisitions from last 12 months: {len(active_recent_reqs)}")

# Show breakdown by status
import collections
status_breakdown = collections.Counter([r.get("status") for r in recent_reqs])
print(f"\nStatus breakdown (last 12 months):")
for status, count in status_breakdown.items():
    print(f"  {status}: {count}")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from datetime import datetime, timedelta

all_line_items = []
six_months_ago = datetime.now() - timedelta(days=180)

filtered_reqs = [
    r for r in all_requisitions
    if r.get("updated_at") and
    datetime.strptime(r["updated_at"][:10], "%Y-%m-%d") >= six_months_ago
]

print(f"Pulling line items for {len(filtered_reqs)} requisitions from last 6 months...")

for req in filtered_reqs:
    req_id = req["id"]
    project_id = req["project_id"]
    project_name = req["project_name"]
    req_number = req.get("number", "")
    req_status = req.get("status", "")

    max_retries = 3
    retry_count = 0

    while retry_count < max_retries:
        response = requests.get(
            f"https://api.procore.com/rest/v1.0/requisitions/{req_id}/contract_detail_items",
            headers=headers,
            params={"project_id": project_id}
        )

        if response.status_code == 429:
            print(f"  Rate limited — waiting 30 seconds...")
            time.sleep(30)
            retry_count += 1
            continue

        if response.status_code != 200:
            print(f"  Error {response.status_code} for requisition {req_id}, skipping")
            break

        rows = response.json()

        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["requisition_id"] = req_id
            row["requisition_number"] = req_number
            row["requisition_status"] = req_status
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_line_items.extend(rows)
        break

    time.sleep(1)

print(f"Done! Total line items: {len(all_line_items)}")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# Test one approved requisition with different endpoints
test_req = next(r for r in all_requisitions if r.get("status") == "approved")
test_req_id = test_req["id"]
test_project_id = test_req["project_id"]

print(f"Testing requisition {test_req_id} on project {test_project_id}")

# Try contract_detail_items instead
response = requests.get(
    f"https://api.procore.com/rest/v1.0/requisitions/{test_req_id}/contract_detail_items",
    headers=headers,
    params={"project_id": test_project_id}
)
print(f"contract_detail_items: {response.status_code} - {len(response.json()) if isinstance(response.json(), list) else response.json()}")

# Try contract_items
response2 = requests.get(
    f"https://api.procore.com/rest/v1.0/requisitions/{test_req_id}/contract_items",
    headers=headers,
    params={"project_id": test_project_id}
)
print(f"contract_items: {response2.status_code} - {len(response2.json()) if isinstance(response2.json(), list) else response2.json()}")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# Check requisition statuses
import collections

statuses = collections.Counter([r.get("status", "unknown") for r in all_requisitions])
print("Requisition statuses:")
for status, count in statuses.items():
    print(f"  {status}: {count}")

# Check how many had errors vs returned data
print(f"\nTotal requisitions: {len(all_requisitions)}")
print(f"Total line items returned: {len(all_line_items)}")
print(f"Average lines per requisition: {len(all_line_items)/len(all_requisitions):.1f}")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
import pandas as pd
import re

clean_rows = []
for row in all_line_items:
    clean_row = {}
    for key, value in row.items():
        if value is None:
            clean_row[key] = None
        elif isinstance(value, (dict, list)):
            clean_row[key] = json.dumps(value)
        elif isinstance(value, bool):
            clean_row[key] = str(value)
        elif isinstance(value, (int, float, str)):
            clean_row[key] = value
        else:
            clean_row[key] = str(value)
    clean_rows.append(clean_row)

def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

pdf = pd.DataFrame(clean_rows)
pdf.columns = [clean_column_name(c) for c in pdf.columns]

for col in pdf.columns:
    if pdf[col].dtype == object:
        pdf[col] = pdf[col].astype(str).replace('None', None)

spark.sql("DROP TABLE IF EXISTS procore_requisition_line_items_raw")

df = spark.createDataFrame(pdf)
df.write.format("delta").mode("append").saveAsTable("procore_requisition_line_items_raw")

print("Saved to Bronze_Lakehouse successfully")

StatementMeta(, , -1, Waiting, , Waiting, True)